# Bài học 13: Xây Dựng Mini Pipeline

## Từ Agent Đơn Giản Đến Pipeline Thực Tế

Ở Bài học 12, bạn đã nối 2 agent bằng cách truyền text đơn giản. Trong Mô-đun 4, bạn sẽ thấy pipeline production với nested schema, theo dõi database, và xử lý lỗi.

Bài học này **là cầu nối giữa hai phần đó**. Chúng ta sẽ xây dựng mini pipeline 3 agent, giới thiệu:
- **Nested schema** — danh sách các object nằm bên trong object khác (khái niệm then chốt cho Mô-đun 4)
- **Instruction dài hơn, chính xác hơn** — gần với chất lượng production
- **Pattern sys.path.insert** — cách notebook import code production

Khi kết thúc bài này, bạn sẽ thoải mái đọc code pipeline production.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

from agno.agent import Agent
from agno.models.anthropic import Claude
from agno.tools.duckduckgo import DuckDuckGoTools
from pydantic import BaseModel, Field

## Xây Dựng Schema Từng Bước

Ở Bài học 11, bạn đã dùng một schema đơn giản:

```python
class ArticleOutline(BaseModel):
    title: str
    sections: list[str]      # ← danh sách các chuỗi
    target_keyword: str
```

Schema đó hoạt động được, nhưng outline thực tế cần **nhiều chi tiết hơn cho mỗi section**. Nếu mỗi section cần heading riêng, key point riêng, VÀ keyword riêng thì sao?

Chúng ta cần **nested schema** — một model nằm bên trong model khác.

In [ ]:
# Bước 1: Những gì bạn đã biết (từ Bài học 11)
class SimpleOutline(BaseModel):
    title: str = Field(description="Article title")
    sections: list[str] = Field(description="Section headings")

print("SimpleOutline — mỗi section chỉ là một chuỗi:")
print('  sections: ["Introduction", "Main Topic", "Conclusion"]')
print()

# Bước 2: MỚI — Định nghĩa Section như một model riêng
class Section(BaseModel):
    heading: str = Field(description="Section heading (H2)")
    key_points: list[str] = Field(description="Key points to cover in this section")
    keywords: list[str] = Field(default_factory=list, description="Target keywords for this section")

# Bước 3: Dùng Section bên trong outline
class DetailedOutline(BaseModel):
    title: str = Field(description="SEO-optimized article title")
    target_keyword: str = Field(description="Primary target keyword")
    sections: list[Section] = Field(description="Ordered list of article sections")

print("DetailedOutline — mỗi section là một object đầy đủ:")
print('  sections: [')
print('    Section(heading="Introduction", key_points=["What is SEO", "Why it matters"], keywords=["SEO"]),')
print('    Section(heading="On-Page Factors", key_points=["Title tags", "Meta descriptions"], keywords=["on-page SEO"]),')
print('  ]')
print()
print("Điểm khác biệt mấu chốt: list[str] vs list[Section]")
print("  list[str]     = danh sách các chuỗi text thuần")
print("  list[Section] = danh sách mà mỗi phần tử là một object Section với các field riêng")

## Hiểu Về Nested Schema

So sánh trực quan:

```
SimpleOutline                    DetailedOutline
├── title: str                   ├── title: str
├── sections: list[str]          ├── target_keyword: str
│   ├── "Introduction"           └── sections: list[Section]
│   ├── "Main Topic"                 ├── Section
│   └── "Conclusion"                 │   ├── heading: str
│                                    │   ├── key_points: list[str]
│                                    │   └── keywords: list[str]
│                                    ├── Section
│                                    │   ├── heading: ...
│                                    │   └── ...
│                                    └── ...
```

`ContentOutline` production trong `agents/schemas.py` dùng đúng pattern này — `OutlineSection` là nested model, giống như `Section` của chúng ta ở trên.

## Xây Dựng 3 Agent

Bây giờ chúng ta xây mini pipeline với 3 agent:
1. **Researcher** — tìm kiếm trên web, trả về ghi chú nghiên cứu (plain text)
2. **Outliner** — nhận nghiên cứu, trả về `DetailedOutline` (nested schema!)
3. **Writer** — nhận outline, viết bài ngắn (Markdown thuần)

Lưu ý: instruction **dài hơn và chính xác hơn** so với Bài học 8-12. Đây là mức gần với production.

> **Chi phí:** ~$0.20-0.40 cho cả ba agent (Claude Sonnet). Mất khoảng 1-2 phút.

In [ ]:
# Agent 1: Researcher — tìm kiếm trên web
researcher = Agent(
    name="Mini Researcher",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    tools=[DuckDuckGoTools()],
    instructions=[
        "You are an expert SEO researcher.",
        "Research the given topic using web search.",
        "Find primary keywords, secondary keywords, and what competitors cover.",
        "Identify content gaps — what's missing from existing articles.",
        "Return organized research notes with clear sections.",
    ],
)

# Agent 2: Outliner — tạo outline có cấu trúc
outliner = Agent(
    name="Mini Outliner",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    output_schema=DetailedOutline,
    instructions=[
        "You are an expert content strategist.",
        "Given research notes, create a structured article outline.",
        "Include 4-6 sections with clear H2 headings.",
        "Each section must have 2-4 key points and at least 1 target keyword.",
        "The title should be SEO-optimized and compelling.",
    ],
)

# Agent 3: Writer — viết bài từ outline
writer = Agent(
    name="Mini Writer",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    instructions=[
        "You are an expert SEO content writer.",
        "Write a concise article (500-800 words) following the provided outline exactly.",
        "Use Markdown: H1 for title, H2 for sections.",
        "Cover every key point listed in each section.",
        "Naturally incorporate the target keywords.",
        "Write in a clear, professional tone.",
    ],
    markdown=True,
)

print("Đã tạo 3 agent: Researcher, Outliner, Writer")

In [ ]:
topic = "How to write SEO-friendly blog titles"

# Bước 1: Nghiên cứu
print("Bước 1: Đang nghiên cứu...")
research = researcher.run(f"Research this topic for an SEO article: {topic}")
print(f"  Xong! ({len(research.content)} ký tự ghi chú nghiên cứu)\n")

# Bước 2: Outline (với nested schema!)
print("Bước 2: Đang tạo outline...")
outline_response = outliner.run(
    f"Create a detailed article outline from these research notes:\n\n{research.content}"
)
outline = outline_response.content
print(f"  Tiêu đề: {outline.title}")
print(f"  Keyword: {outline.target_keyword}")
print(f"  Số section: {len(outline.sections)}\n")

# Bước 3: Truy cập dữ liệu lồng nhau — đây là kỹ năng mấu chốt!
print("Truy cập dữ liệu lồng nhau:")
for i, section in enumerate(outline.sections, 1):
    print(f"  Section {i}: {section.heading}")
    print(f"    Key points: {section.key_points}")
    print(f"    Keywords: {section.keywords}")
    print()

In [ ]:
# Bước 3: Viết bài từ outline
print("Bước 3: Đang viết bài...")

# Chuyển outline thành chuỗi cho writer
outline_text = f"Title: {outline.title}\nKeyword: {outline.target_keyword}\n\n"
for section in outline.sections:
    outline_text += f"## {section.heading}\n"
    outline_text += f"Key points: {', '.join(section.key_points)}\n"
    outline_text += f"Keywords: {', '.join(section.keywords)}\n\n"

article = writer.run(f"Write an article from this outline:\n\n{outline_text}")
print(f"  Xong! ({len(article.content.split())} từ)\n")
print("--- Xem trước bài viết (500 ký tự đầu) ---")
print(article.content[:500])
print("\n...")

## Đọc Code Production

Bây giờ hãy so sánh mini pipeline của chúng ta với pipeline thật. Pattern hoàn toàn giống nhau:

```python
# Mini pipeline (vừa xây xong)                 # Pipeline thật (pipeline.py)
research = researcher.run(topic)               research = research_agent.run(topic)
outline = outliner.run(research.content)        outline = outline_agent.run(research.content)
article = writer.run(outline_text)              article = writer_agent.run(outline_json)
```

Pipeline thật bổ sung:
- **Theo dõi database** — `update_article_status(id, "writing")`
- **Xử lý lỗi** — `try/except` bao bọc mỗi bước
- **Xuất file** — lưu bài viết thành file `.md`
- **Bổ sung hình ảnh** — bước agent thứ 4 (tùy chọn)
- **Grok cho viết bài** — dùng model khác để viết long-form tốt hơn

## Import Code Production Từ Notebook

Ô tiếp theo sử dụng `sys.path.insert(0, '../../output')`. Cách nó hoạt động như sau:

Khi Python import một module (như `from agents import research_agent`), nó tìm trong danh sách các thư mục gọi là **đường dẫn tìm kiếm module**. Mặc định, notebook chỉ tìm trong thư mục của chính nó.

Code production của chúng ta nằm trong `output/` — hai cấp thư mục phía trên notebook này. `sys.path.insert(0, '../../output')` thêm thư mục đó vào đường dẫn tìm kiếm của Python, để các lệnh import như `from agents import ...` hoạt động được.

Bạn sẽ thấy cùng pattern này trong Bài học 16-19 và 21 — mọi notebook dùng code production đều cần dòng này.

In [ ]:
import sys, os

# Dòng này nói với Python: "hãy tìm thêm trong thư mục output/ khi import"
# Cần thiết vì notebook nằm ở lessons_vi/, không phải output/
sys.path.insert(0, os.path.abspath("../../output"))

# Giờ chúng ta có thể import code production!
from agents import ContentOutline, OutlineSection

print("Schema ContentOutline production:")
print(f"  Các field: {list(ContentOutline.model_fields.keys())}")
print()
print("Schema OutlineSection production:")
print(f"  Các field: {list(OutlineSection.model_fields.keys())}")
print()
print("So sánh với DetailedOutline của chúng ta:")
print(f"  Các field: {list(DetailedOutline.model_fields.keys())}")
print()
print("So sánh với Section của chúng ta:")
print(f"  Các field: {list(Section.model_fields.keys())}")
print()
print("Cùng một pattern! Phiên bản production chỉ có nhiều field hơn.")

## So Sánh Mini vs Production Pipeline

| Tính năng | Mini Pipeline (bài này) | Pipeline Thật (pipeline.py) |
|-----------|------------------------|----------------------------|
| Agent | 3 (researcher, outliner, writer) | 4 (+ image agent) |
| Schema | `DetailedOutline` với `Section` | `ContentOutline` với `OutlineSection` |
| Model viết bài | Claude Sonnet | Grok-4 (viết tốt hơn) |
| Database | Không có | Theo dõi trạng thái bằng Airtable |
| Xử lý lỗi | Không có | try/except ở mỗi bước |
| Xuất file | Không có | Lưu .md vào content/ |
| Chi phí | ~$0.20-0.40 | ~$1-3 |

**Pattern** là giống nhau. Phiên bản production thêm độ ổn định và tính năng phục vụ thực tế.

## Tổng Kết Bài Học 13

Những gì bạn đã học:
- **Nested schema** — `list[Section]` thay vì `list[str]` để dữ liệu phong phú hơn
- **Truy cập dữ liệu lồng nhau** — `outline.sections[0].heading`, `outline.sections[0].key_points`
- **Pipeline 3 agent** — researcher -> outliner -> writer, mỗi agent có instruction tập trung
- **`sys.path.insert`** — cách notebook import code production từ `output/`
- **Mini vs pipeline thật** — cùng pattern, production thêm theo dõi Airtable, xử lý lỗi, xuất file

### Tổng Kết Mô-đun 3 — Xây Dựng Agent

| Bài học | Khái niệm |
|---------|----------|
| Bài học 8 | Tạo agent cơ bản với name, model, instructions |
| Bài học 9 | Thêm tools — agent có thể tìm kiếm trên web |
| Bài học 11 | Structured output — buộc agent trả về dữ liệu có cấu trúc |
| Bài học 12 | Nối agent — output của agent này thành input cho agent kia |
| Bài học 13 | Mini pipeline — nested schema, chuỗi 3 agent, pattern production |

**Mô-đun tiếp theo:** Mô-đun 4 — Pipeline SEO Thực Tế. Bạn sẽ thấy agent production với schema đầy đủ, Grok cho viết bài, bổ sung hình ảnh, và pipeline hoàn chỉnh với theo dõi database.

## Bài Tập

Chỉnh sửa schema `DetailedOutline`:
1. Thêm field `tone` (str, description "Writing tone for the article, e.g., 'professional', 'casual'")
2. Thêm field `target_audience` (str, description "Who this article is written for")
3. Chạy lại outliner với schema đã cập nhật
4. Kiểm tra các field mới đã được điền: `outline.tone`, `outline.target_audience`

Đây chính là cách bạn tùy chỉnh schema production — thêm field, chạy lại, kiểm tra.

In [ ]:
# Bước 1: Định nghĩa schema mở rộng
class ExtendedOutline(___):
    title: str = Field(description="SEO-optimized article title")
    target_keyword: str = Field(description="Primary target keyword")
    sections: list[Section] = Field(description="Ordered list of article sections")
    tone: ___ = Field(description="Writing tone for the article, e.g., 'professional', 'casual'")
    target_audience: ___ = Field(description="Who this article is written for")

# Bước 2: Tạo outliner mới với schema đã cập nhật
extended_outliner = Agent(
    name="Extended Outliner",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    output_schema=___,
    instructions=[
        "You are an expert content strategist.",
        "Given research notes, create a structured article outline.",
        "Include 4-6 sections with clear H2 headings.",
        "Each section must have 2-4 key points and at least 1 target keyword.",
        "Also specify the writing tone and target audience.",
    ],
)

# Bước 3: Chạy với research đã có
extended_response = extended_outliner.run(
    f"Create a detailed article outline from these research notes:\n\n{research.___}"
)
extended_outline = extended_response.___

# Bước 4: Kiểm tra các field mới
print(f"Title: {extended_outline.title}")
print(f"Tone: {extended_outline.___}")
print(f"Target audience: {extended_outline.___}")

<details>
<summary>Bấm để xem đáp án</summary>

```python
class ExtendedOutline(BaseModel):
    title: str = Field(description="SEO-optimized article title")
    target_keyword: str = Field(description="Primary target keyword")
    sections: list[Section] = Field(description="Ordered list of article sections")
    tone: str = Field(description="Writing tone for the article, e.g., 'professional', 'casual'")
    target_audience: str = Field(description="Who this article is written for")

extended_outliner = Agent(
    name="Extended Outliner",
    model=Claude(id="claude-sonnet-4-5-20250929"),
    output_schema=ExtendedOutline,
    instructions=[
        "You are an expert content strategist.",
        "Given research notes, create a structured article outline.",
        "Include 4-6 sections with clear H2 headings.",
        "Each section must have 2-4 key points and at least 1 target keyword.",
        "Also specify the writing tone and target audience.",
    ],
)

extended_response = extended_outliner.run(
    f"Create a detailed article outline from these research notes:\n\n{research.content}"
)
extended_outline = extended_response.content

print(f"Title: {extended_outline.title}")
print(f"Tone: {extended_outline.tone}")
print(f"Target audience: {extended_outline.target_audience}")
```
</details>

## Sẵn sàng cho Module 4? — Kiểm tra nhanh

Trước khi tiếp tục, hãy chắc chắn bạn hiểu 3 khái niệm chính này. Hãy thử trả lời mà không xem lại bài.

**1. Sự khác biệt giữa `list[str]` và `list[Section]` là gì?**

<details>
<summary>Bấm để xem đáp án</summary>

`list[str]` là danh sách chuỗi văn bản đơn giản. `list[Section]` là danh sách mà mỗi phần tử là một đối tượng Section với các trường riêng (heading, key_points, keywords). Schema lồng nhau cho dữ liệu phong phú hơn.
</details>

**2. Tại sao writer agent dùng Grok thay vì Claude trong pipeline thật?**

<details>
<summary>Bấm để xem đáp án</summary>

Grok viết văn bản dài rất tốt và tự nhiên. Writer agent chỉ cần xuất Markdown thuần — không cần tools, không cần structured output. Vì Grok không thể kết hợp tools + output_schema, dùng nó cho bước chỉ viết là phù hợp nhất.
</details>

**3. `sys.path.insert(0, '../../output')` làm gì?**

<details>
<summary>Bấm để xem đáp án</summary>

Nó thêm thư mục `output/` vào đường dẫn tìm kiếm module của Python. Không có dòng này, notebook trong `lessons_vi/` không thể import code production. Mọi notebook dùng code production đều cần dòng này.
</details>